# Notebook 42 — L1 Pendulum: Unified Held-out Evaluation
**Runtime:** Colab CPU (no torch needed). **Prerequisites:** 39, then completed 40/41 runs.

Each L1 condition compares all completed compatible runs. Missing methods are reported;
they are never silently labelled as evaluated. PCA, scalers and probes fit **train** only;
ridge hyperparameters use **validation** only. Test groups supply final scores and
grouped bootstrap intervals. Intervals condition on each trained model; seeds stay
separate. Avoid repeatedly selecting methods/hyperparameters on these test scores.

Outputs: theta/velocity/(L) readouts, optional fixed-capacity nonlinear readouts,
direct forecasts at ~T0/4, T0, 2T0, train-only ID diagnostics, latent plots and CSVs.
Direct ridge forecasts do not constitute autonomous rollouts or prove a nonlinear
latent is dynamically insufficient. The oracle-state persistence baseline is labelled
because it receives the true current theta/velocity. Delay-PCA receives observed windows.


## Cell 1 — Drive and isolated paths


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, tempfile, json

PROJECT_DIR = Path('/content/drive/MyDrive/BSISO_SSL_Project')
L1_ROOT = PROJECT_DIR / 'toy' / 'pendulum'
L1_ROOT.mkdir(parents=True, exist_ok=True)
# Embedded code needs no GitHub clone/download or external .py upload.
RUNTIME_DIR = Path(tempfile.mkdtemp(prefix='ddcs_l1_code_'))
os.chdir(RUNTIME_DIR)
sys.path.insert(0, str(RUNTIME_DIR))
for name in ('l1_data', 'l1_learning', 'l1_evaluation'):
    sys.modules.pop(name, None)
CACHE_DIR = Path('/content/ddcs_l1_frames')  # disposable; never store frame arrays on Drive

def resolve_dataset_id(value):
    if value:
        if not (L1_ROOT / 'datasets' / value / 'manifest.json').exists():
            raise FileNotFoundError(f'Dataset {value} not found; run notebook 39 first.')
        return value
    available = sorted(p.parent.name for p in (L1_ROOT / 'datasets').glob('*/manifest.json'))
    if len(available) != 1:
        raise ValueError(f'Set DATASET_ID explicitly from notebook 39. Available: {available}')
    return available[0]

print('Drive outputs:', L1_ROOT)
print('Local code:', RUNTIME_DIR)


## Cell 2 — Embedded CPU implementations


In [ ]:
%%writefile l1_data.py
"""L1 pendulum simulation, observation, split and provenance utilities (no torch)."""
from __future__ import annotations

from dataclasses import asdict, dataclass
import hashlib
import json
from pathlib import Path

import numpy as np

SCHEMA = "pendulum-l1-v1"
SPLITS = {"train": 0, "val": 1, "test": 2}


def json_write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
    tmp.replace(path)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()[:16]


def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


@dataclass(frozen=True)
class SimulationConfig:
    seed: int = 20261027
    n_fixed: int = 100
    n_length_groups: int = 60
    trajectories_per_length: int = 5
    periods: float = 8.0
    frames_per_period: int = 50
    substeps: int = 8
    g: float = 1.0
    reference_length: float = 1.0
    damping: float = 0.02
    length_min: float = 0.8
    length_max: float = 1.2
    image_size: int = 64

    def validate(self):
        if self.n_fixed < 5 or self.n_length_groups < 5:
            raise ValueError("Need at least five independent groups for train/val/test.")
        if self.trajectories_per_length < 1 or self.periods <= 0:
            raise ValueError("Invalid trajectory count/duration.")
        if self.frames_per_period < 10 or self.substeps < 1:
            raise ValueError("Insufficient sampling/integration resolution.")
        if self.g <= 0 or self.reference_length <= 0 or self.damping < 0:
            raise ValueError("Invalid physical parameters.")
        if not 0 < self.length_min < self.length_max <= 1.5 * self.reference_length:
            raise ValueError("Length interval must be positive and fit the fixed camera.")
        if self.image_size != 64:
            raise ValueError("The initial CNN/renderer protocol uses 64 x 64 images.")

    @property
    def period(self):
        return 2 * np.pi * np.sqrt(self.reference_length / self.g)

    @property
    def dt(self):
        return self.period / self.frames_per_period


def group_split(groups, seed):
    groups = np.asarray(groups)
    unique = np.unique(groups)
    if len(unique) < 5:
        raise ValueError("Too few independent groups.")
    order = np.random.default_rng(seed).permutation(unique)
    n_val = max(1, int(round(0.2 * len(order))))
    n_test = n_val
    mapping = {int(g): 0 for g in order}
    mapping.update({int(g): 1 for g in order[-n_val-n_test:-n_test]})
    mapping.update({int(g): 2 for g in order[-n_test:]})
    return np.array([mapping[int(g)] for g in groups], dtype=np.int8)


def simulate(initial, lengths, times, g=1.0, damping=0.02, substeps=8):
    """Vectorized RK4, constant length per trajectory; no stochastic force in L1a-c."""
    state = np.asarray(initial, dtype=np.float64).copy()
    lengths = np.asarray(lengths, dtype=np.float64)
    times = np.asarray(times)
    if state.shape != (len(lengths), 2) or np.any(lengths <= 0):
        raise ValueError("Expected initial=(trajectories,2), positive lengths.")
    if substeps < 1 or len(times) < 2 or np.any(np.diff(times) <= 0):
        raise ValueError("Invalid time grid or substeps.")
    out = np.empty((len(lengths), len(times), 2), dtype=np.float64)
    out[:, 0] = state

    def rhs(y):
        return np.stack((y[:, 1], -damping*y[:, 1] - g/lengths*np.sin(y[:, 0])), axis=1)

    for i, interval in enumerate(np.diff(times), start=1):
        h = interval / substeps
        for _ in range(substeps):
            k1 = rhs(state)
            k2 = rhs(state + h*k1/2)
            k3 = rhs(state + h*k2/2)
            k4 = rhs(state + h*k3)
            state += h*(k1 + 2*k2 + 2*k3 + k4)/6
        out[:, i] = state
    return out


def energy(states, lengths, g=1.0):
    """Mechanical energy for unit mass (constant L), not a harmonic approximation."""
    length = np.asarray(lengths)[:, None]
    return 0.5*length**2*states[..., 1]**2 + g*length*(1-np.cos(states[..., 0]))


def make_bundle(cfg, variable_length=False):
    cfg.validate()
    rng = np.random.default_rng(cfg.seed + (1000 if variable_length else 0))
    if variable_length:
        group_lengths = rng.uniform(cfg.length_min, cfg.length_max, cfg.n_length_groups)
        groups = np.repeat(np.arange(cfg.n_length_groups), cfg.trajectories_per_length)
        lengths = group_lengths[groups]
    else:
        groups = np.arange(cfg.n_fixed)
        lengths = np.full(cfg.n_fixed, cfg.reference_length)
    initial = rng.uniform([-1.0, -0.5], [1.0, 0.5], size=(len(groups), 2))
    # Avoid a dataset dominated by almost stationary initial conditions.
    weak = np.linalg.norm(initial, axis=1) < 0.15
    while weak.any():
        initial[weak] = rng.uniform([-1.0, -0.5], [1.0, 0.5], size=(int(weak.sum()), 2))
        weak = np.linalg.norm(initial, axis=1) < 0.15
    times = np.arange(int(round(cfg.periods*cfg.frames_per_period)) + 1)*cfg.dt
    return {
        "states": simulate(initial, lengths, times, cfg.g, cfg.damping, cfg.substeps),
        "times": times, "lengths": lengths, "groups": groups.astype(np.int32),
        "splits": group_split(groups, cfg.seed + 17),
    }


def numerical_checks(bundle, cfg):
    original = bundle["states"][:3]
    fine = simulate(original[:, 0], bundle["lengths"][:3], bundle["times"],
                    cfg.g, cfg.damping, cfg.substeps*2)
    err = float(np.max(np.abs(original-fine)))
    e = energy(bundle["states"], bundle["lengths"], cfg.g)
    max_rise = float(np.max(np.diff(e, axis=1)))
    if err > 1e-5 or max_rise > 1e-8:
        raise ValueError(f"Integrator check failed: convergence={err}, energy rise={max_rise}")
    for group in np.unique(bundle["groups"]):
        if len(np.unique(bundle["splits"][bundle["groups"] == group])) != 1:
            raise ValueError("A group crossed splits.")
    return {"rk4_step_halving_max_error": err, "max_energy_increment": max_rise,
            "n_trajectories": len(e), "frames_per_trajectory": e.shape[1]}


def create_dataset(root, cfg):
    """Small truth arrays on Drive; frames are rendered only into disposable Colab cache."""
    cfg.validate()
    identity = {"schema": SCHEMA, "simulation": asdict(cfg), "source_sha": file_sha(__file__)}
    dataset_id = digest(identity)
    out = Path(root)/"datasets"/dataset_id
    if (out/"manifest.json").exists():
        load_dataset(root, dataset_id, "fixed")  # Validate existing hashes, never overwrite.
        load_dataset(root, dataset_id, "variable")
        return dataset_id
    if out.exists():
        raise FileExistsError(f"Incomplete dataset exists: {out}; use a new seed or inspect it.")
    out.mkdir(parents=True)
    checks, hashes = {}, {}
    for name, variable in [("fixed", False), ("variable", True)]:
        bundle = make_bundle(cfg, variable)
        checks[name] = numerical_checks(bundle, cfg)
        np.savez_compressed(out/f"{name}.npz", **bundle)
        hashes[name] = file_sha(out/f"{name}.npz")
    json_write(out/"manifest.json", {**identity, "dataset_id": dataset_id,
                                    "checks": checks, "data_sha256": hashes})
    return dataset_id


def load_dataset(root, dataset_id, kind):
    if kind not in {"fixed", "variable"}:
        raise ValueError(kind)
    out = Path(root)/"datasets"/dataset_id
    manifest = json.loads((out/"manifest.json").read_text())
    if manifest["schema"] != SCHEMA or manifest["dataset_id"] != dataset_id:
        raise ValueError("Dataset identity/schema mismatch.")
    if manifest["source_sha"] != file_sha(__file__):
        raise ValueError("Data code differs from notebook 39; use the same notebook release.")
    if file_sha(out/f"{kind}.npz") != manifest["data_sha256"][kind]:
        raise ValueError("Dataset file checksum mismatch.")
    with np.load(out/f"{kind}.npz", allow_pickle=False) as data:
        bundle = {k: data[k] for k in data.files}
    return bundle, SimulationConfig(**manifest["simulation"]), manifest


CONDITIONS = {
    "L1a": {"kind": "fixed", "noise_std": 0.0, "visible_length": True},
    "L1b_s002": {"kind": "fixed", "noise_std": 0.02, "visible_length": True},
    "L1b_s005": {"kind": "fixed", "noise_std": 0.05, "visible_length": True},
    "L1c_visible": {"kind": "variable", "noise_std": 0.02, "visible_length": True},
    "L1c_hidden": {"kind": "variable", "noise_std": 0.02, "visible_length": False},
}


def render(theta, lengths, size=64, visible_length=True, reference_length=1.0):
    """Fixed camera, Gaussian anti-aliased rod and bob; returns (N,H,W) in [0,1]."""
    theta = np.atleast_1d(theta)
    lengths = np.broadcast_to(lengths, theta.shape)
    radius = 0.42*(size-1)*(lengths/reference_length if visible_length else np.ones_like(theta))
    px, py = (size-1)/2, 0.14*(size-1)
    dx, dy = radius*np.sin(theta), radius*np.cos(theta)
    yy, xx = np.mgrid[:size, :size]
    rx, ry = xx[None]-px, yy[None]-py
    a = np.clip((rx*dx[:, None, None]+ry*dy[:, None, None]) /
                radius[:, None, None]**2, 0, 1)
    d2 = (rx-a*dx[:, None, None])**2+(ry-a*dy[:, None, None])**2
    rod = 0.45*np.exp(-d2/(2*0.65**2))
    bob = np.exp(-((rx-dx[:, None, None])**2+(ry-dy[:, None, None])**2)/(2*1.65**2))
    pivot = 0.7*np.exp(-(rx**2+ry**2)/(2*1.1**2))
    return np.maximum(np.maximum(rod, bob), pivot).astype(np.float32)


def frame_cache(bundle, cfg, condition, cache_root, dataset_id):
    spec = CONDITIONS[condition]
    name = f"{dataset_id}_{spec['kind']}_{'visible' if spec['visible_length'] else 'hidden'}"
    path = Path(cache_root)/f"{name}.npy"
    shape = (*bundle["states"].shape[:2], cfg.image_size, cfg.image_size)
    if path.exists():
        arr = np.load(path, mmap_mode="r", allow_pickle=False)
        if arr.shape != shape or arr.dtype != np.uint8:
            raise ValueError("Frame cache shape/dtype mismatch; remove this disposable cache.")
        return arr
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".partial.npy")
    arr = np.lib.format.open_memmap(tmp, mode="w+", dtype="uint8", shape=shape)
    for tr in range(shape[0]):
        arr[tr] = np.rint(255*render(bundle["states"][tr, :, 0], bundle["lengths"][tr],
                                   cfg.image_size, spec["visible_length"], cfg.reference_length)).astype("uint8")
    arr.flush()
    del arr
    tmp.replace(path)
    return np.load(path, mmap_mode="r", allow_pickle=False)


def index_windows(bundle, window=5, future=5, stride=2):
    """Each row is (trajectory, final input frame); all windows remain in one trajectory."""
    if min(window, stride) < 1 or future < 0:
        raise ValueError("Invalid window/stride/future.")
    ends = np.arange(window-1, bundle["states"].shape[1]-future, stride)
    if not len(ends):
        raise ValueError("Trajectories are too short for the requested history/horizon.")
    n = len(bundle["states"])
    return np.column_stack((np.repeat(np.arange(n), len(ends)), np.tile(ends, n))).astype("int32")


def observation(frames, trajectory, end, window, noise_std, seed, draw=0):
    """Noise is keyed by frame: overlapping windows share the same observed pixels."""
    start = int(end)-window+1
    if start < 0 or end >= frames.shape[1]:
        raise IndexError("Window outside trajectory.")
    out = np.array(frames[int(trajectory), start:int(end)+1], dtype="float32")/255
    if noise_std:
        for j, t in enumerate(range(start, int(end)+1)):
            rng = np.random.default_rng(np.random.SeedSequence([seed, draw, int(trajectory), t]))
            out[j] += noise_std*rng.standard_normal(out[j].shape).astype("float32")
    # Do not clip or normalize individual frames: keep the stated zero-mean noise model.
    return out


def sample_truth(bundle, index, horizon=0):
    tr, t = index.T
    return np.column_stack((bundle["states"][tr, t+horizon], bundle["lengths"][tr]))


def estimate_id(features, groups, ks=(10, 20, 40), max_points=2000, seed=17):
    """Train-only kNN MLE diagnostic; exclude neighbors from the same trajectory/group.

    This is a finite-sample, scale-dependent estimate, not a guaranteed physical dimension.
    No per-coordinate whitening: it would amplify near-null latent directions.
    """
    from sklearn.neighbors import NearestNeighbors
    z = np.asarray(features, dtype="float64")
    groups = np.asarray(groups)
    if len(z) != len(groups) or z.ndim != 2 or not np.isfinite(z).all():
        raise ValueError("Invalid ID input.")
    take = np.random.default_rng(seed).choice(len(z), min(len(z), max_points), replace=False)
    z, groups = z[take], groups[take]
    candidates = min(len(z), max(128, max(ks)*8))
    distances, neighbors = NearestNeighbors(n_neighbors=candidates).fit(z).kneighbors(z)
    rows = []
    for k in ks:
        estimates = []
        for i in range(len(z)):
            valid = (groups[neighbors[i]] != groups[i]) & (distances[i] > 1e-10)
            d = distances[i][valid][:k]
            if len(d) == k:
                denominator = np.log(d[-1]/d[:-1]).sum()
                if denominator > 1e-10:
                    estimates.append((k-2)/denominator)
        rows.append({"k": int(k), "n_valid": len(estimates),
                     "median_id": float(np.median(estimates)) if estimates else None,
                     "mean_id": float(np.mean(estimates)) if estimates else None})
    singular = np.linalg.svd(z-z.mean(0), compute_uv=False)**2
    pr = float(singular.sum()**2/(np.square(singular).sum()+1e-30))
    return {"n_sampled": len(z), "scales": rows, "participation_ratio": pr,
            "neighbor_exclusion": "same group", "used_split": "train"}


In [ ]:
%%writefile l1_evaluation.py
"""CPU evaluation: train-only probes/PCA, common direct forecasts, grouped uncertainty."""
from __future__ import annotations

from dataclasses import asdict, dataclass
import json
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

from l1_data import (CONDITIONS, digest, estimate_id, file_sha, frame_cache, index_windows,
                     json_write, load_dataset, observation, sample_truth)


@dataclass(frozen=True)
class EvalConfig:
    seed: int = 123
    window: int = 5
    stride: int = 4
    pca_components: int = 64
    pca_fit_samples: int = 3000
    bootstrap_repeats: int = 300
    nonlinear_probe: bool = True
    nonlinear_max_iter: int = 200


def ridge_probe(x, y, splits):
    """Fit all scalers on train; select ridge alpha using validation only."""
    train, val = splits == 0, splits == 1
    scaler = StandardScaler().fit(x[train])
    xx = scaler.transform(x)
    mean, scale = y[train].mean(0), np.maximum(y[train].std(0), 1e-8)
    yy = (y-mean)/scale
    best = None
    for alpha in (1e-4, 0.01, 1.0, 100.0, 10000.0):
        model = Ridge(alpha=alpha).fit(xx[train], yy[train])
        score = float(np.mean((model.predict(xx[val])-yy[val])**2))
        if best is None or score < best[0]:
            best = score, model, alpha
    return best[1].predict(xx)*scale+mean, {"alpha": best[2], "val_normalized_mse": best[0]}


def nonlinear_probe(x, y, splits, cfg):
    """Fixed small MLP; no internal random validation split across temporal samples."""
    train, val = splits == 0, splits == 1
    scaler = StandardScaler().fit(x[train])
    xx = scaler.transform(x)
    mean, scale = y[train].mean(0), np.maximum(y[train].std(0), 1e-8)
    yy = (y-mean)/scale
    # One fixed capacity, batch schedule and regularization for every representation.
    model = MLPRegressor(hidden_layer_sizes=(64,), activation="tanh", alpha=0.01,
                         max_iter=cfg.nonlinear_max_iter, batch_size=min(256, int(train.sum())),
                         learning_rate_init=1e-3, early_stopping=False, random_state=cfg.seed)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(xx[train], yy[train])
    info = {"val_normalized_mse": float(np.mean((model.predict(xx[val])-yy[val])**2)),
            "iterations": model.n_iter_,
            "convergence_warning": any(isinstance(w.message, ConvergenceWarning) for w in caught)}
    return model.predict(xx)*scale+mean, info


def metric_rows(y, predicted, columns, groups, repeats=300, seed=123):
    rng = np.random.default_rng(seed)
    unique = np.unique(groups)
    results = []
    for j, name in enumerate(columns):
        error = predicted[:, j]-y[:, j]
        group_mse = np.array([np.mean(error[groups == g]**2) for g in unique])
        boot = np.sqrt(rng.choice(group_mse, size=(repeats, len(unique)), replace=True).mean(1))
        denominator = np.sum((y[:, j]-y[:, j].mean())**2)
        results.append({"variable": name, "rmse": float(np.sqrt(np.mean(error**2))),
                        "mae": float(np.mean(np.abs(error))),
                        "r2": float(1-np.sum(error**2)/denominator) if denominator > 1e-12 else np.nan,
                        "rmse_ci_low": float(np.quantile(boot, 0.025)),
                        "rmse_ci_high": float(np.quantile(boot, 0.975)),
                        "test_groups": len(unique), "test_samples": len(y)})
    return results


def delay_pca(frames, bundle, sim, condition, index, cfg):
    """16x16 block-averaged delay images; fit PCA on train only, then transform all splits."""
    spec = CONDITIONS[condition]
    x = np.empty((len(index), cfg.window*16*16), dtype="float32")
    for i, (tr, t) in enumerate(index):
        clip = observation(frames, tr, t, cfg.window, spec["noise_std"], sim.seed)
        x[i] = clip.reshape(cfg.window, 16, 4, 16, 4).mean(axis=(2, 4)).ravel()
    train = np.flatnonzero(bundle["splits"][index[:, 0]] == 0)
    fit_ids = np.random.default_rng(cfg.seed).choice(train, min(len(train), cfg.pca_fit_samples), replace=False)
    n_components = min(cfg.pca_components, len(fit_ids)-1, x.shape[1])
    if n_components < 1:
        raise ValueError("Insufficient training samples for PCA.")
    pca = PCA(n_components, svd_solver="randomized", random_state=cfg.seed).fit(x[fit_ids])
    return pca.transform(x).astype("float32"), pca.explained_variance_ratio_.tolist()


def completed_runs(root, dataset_id, condition):
    base = Path(root)/"runs"/dataset_id/condition
    return sorted(p.parent for p in base.glob("*/complete.json"))


def compare(root, dataset_id, condition, cache_root, run_paths, cfg=EvalConfig()):
    if cfg.bootstrap_repeats < 1 or cfg.pca_fit_samples < 2:
        raise ValueError("Invalid evaluation sample counts.")
    spec = CONDITIONS[condition]
    bundle, sim, manifest = load_dataset(root, dataset_id, spec["kind"])
    index = index_windows(bundle, cfg.window, 2*sim.frames_per_period, cfg.stride)
    splits, groups = bundle["splits"][index[:, 0]], bundle["groups"][index[:, 0]]
    test = splits == 2
    frames = frame_cache(bundle, sim, condition, cache_root, dataset_id)
    representations, run_meta = {}, []
    for path in map(Path, run_paths):
        meta = json.loads((path/"run.json").read_text())
        complete = json.loads((path/"complete.json").read_text())
        if meta["dataset_id"] != dataset_id or meta["condition"] != condition or meta["condition_spec"] != spec:
            raise ValueError(f"Incompatible dataset/condition: {path}")
        if meta["training"]["window"] != cfg.window or meta["training"]["eval_stride"] != cfg.stride:
            raise ValueError(f"History/evaluation stride mismatch: {path}")
        if file_sha(path/"features.npz") != complete["features_sha256"]:
            raise ValueError(f"Feature checksum mismatch: {path}")
        with np.load(path/"features.npz", allow_pickle=False) as features:
            if not np.array_equal(features["index"], index):
                raise ValueError("Feature rows do not match the common evaluation index.")
            for key in features.files:
                if key == "index":
                    continue
                # Keep training seeds/configs separate; never select the best seed on test.
                label = f"{meta['run_id']}:{key}"
                representations[label] = (features[key].copy(), meta["method"], key, meta["training"]["seed"])
        run_meta.append({"run_id": meta["run_id"], "features_sha": complete["features_sha256"]})
    pca_features, variance = delay_pca(frames, bundle, sim, condition, index, cfg)
    representations["delay_pca"] = (pca_features, "delay_pca", "pca", None)
    identity = {"dataset_id": dataset_id, "condition": condition, "config": asdict(cfg),
                "runs": run_meta, "evaluation_source_sha": file_sha(__file__)}
    out = Path(root)/"analysis"/dataset_id/condition/digest(identity)
    if (out/"complete.json").exists():
        complete = json.loads((out/"complete.json").read_text())
        if file_sha(out/"scorecard.csv") != complete["scorecard_sha256"]:
            raise ValueError("Completed scorecard was modified; inspect the saved analysis.")
        return out, pd.read_csv(out/"scorecard.csv")
    out.mkdir(parents=True, exist_ok=True)
    json_write(out/"evaluation.json", {**identity, "pca_variance_ratio": variance,
               "bootstrap_unit": "length group" if spec["kind"] == "variable" else "trajectory",
               "forecast_type": "direct multi-horizon readout, NOT an autonomous rollout",
               "truth_use": "evaluation probes only; never representation training",
               "uncertainty": "conditional on trained model; training seeds reported separately"})
    truth = sample_truth(bundle, index)
    current_columns = [0, 1, 2] if spec["kind"] == "variable" else [0, 1]
    names = ["theta", "velocity", "length"]
    rows, selections, ids, current_predictions = [], {}, {}, {}
    horizons = sorted(set([0, int(round(sim.frames_per_period/4)), sim.frames_per_period, 2*sim.frames_per_period]))
    for label, (x, method, feature, seed) in representations.items():
        print("Evaluating", label, flush=True)
        if not np.isfinite(x).all():
            raise ValueError(f"Nonfinite representation: {label}")
        ids[label] = estimate_id(x[splits == 0], groups[splits == 0], seed=cfg.seed)
        for horizon in horizons:
            columns = current_columns if horizon == 0 else [0, 1]
            y = sample_truth(bundle, index, horizon)[:, columns]
            predicted, selected = ridge_probe(x, y, splits)
            key = f"{label}:h{horizon}:ridge"
            selections[key] = selected
            if horizon == 0:
                current_predictions[label] = predicted[:, :2]
            for result in metric_rows(y[test], predicted[test], [names[j] for j in columns],
                                      groups[test], cfg.bootstrap_repeats, cfg.seed):
                rows.append({"label": label, "method": method, "feature": feature,
                             "dimension": x.shape[1], "training_seed": seed, "probe": "ridge",
                             "horizon_frames": horizon, "horizon_periods": horizon/sim.frames_per_period,
                             **result})
        if cfg.nonlinear_probe:
            y = truth[:, current_columns]
            predicted, selected = nonlinear_probe(x, y, splits, cfg)
            selections[f"{label}:h0:mlp64"] = selected
            for result in metric_rows(y[test], predicted[test], [names[j] for j in current_columns],
                                      groups[test], cfg.bootstrap_repeats, cfg.seed):
                rows.append({"label": label, "method": method, "feature": feature,
                             "dimension": x.shape[1], "training_seed": seed, "probe": "mlp64",
                             "horizon_frames": 0, "horizon_periods": 0.0, **result})
    # Persistence is explicitly labelled as using the true current physical state.
    for horizon in horizons[1:]:
        future = sample_truth(bundle, index, horizon)[:, :2]
        for result in metric_rows(future[test], truth[test, :2], names[:2], groups[test],
                                  cfg.bootstrap_repeats, cfg.seed):
            rows.append({"label": "oracle_state_persistence", "method": "oracle_state_persistence",
                         "feature": "true_current_state", "dimension": 2, "training_seed": None,
                         "probe": "persistence", "horizon_frames": horizon,
                         "horizon_periods": horizon/sim.frames_per_period, **result})
    scorecard = pd.DataFrame(rows)
    scorecard.to_csv(out/"scorecard.csv", index=False)
    json_write(out/"probe_selection.json", selections)
    json_write(out/"id_train.json", ids)
    np.savez_compressed(out/"evaluation_truth.npz", index=index, states=truth, splits=splits, groups=groups)
    plot_results(out, representations, scorecard, truth, splits, groups, index, current_predictions)
    json_write(out/"complete.json", {"scorecard_sha256": file_sha(out/"scorecard.csv")})
    return out, scorecard


def plot_results(out, representations, scores, truth, splits, groups, index, current_predictions):
    import matplotlib.pyplot as plt
    # Each latent display projection is fitted on train, including when showing test.
    selected = []
    seen = set()
    for label, (x, method, feature, seed) in representations.items():
        if method not in seen and (feature in {"z", "z_d2", "z_d3", "pca"}):
            selected.append((label, x, method, feature))
            seen.add(method)
    colors = [(0, "angle"), (1, "angular velocity")]
    if truth[:, 2].std() > 1e-8:
        colors.append((2, "pendulum length"))
    fig, axes = plt.subplots(len(colors), len(selected),
                             figsize=(4*len(selected), 3.5*len(colors)), squeeze=False)
    test_idx = np.flatnonzero(splits == 2)[::3]
    for col, (label, x, method, feature) in enumerate(selected):
        n = min(2, x.shape[1])
        pca = PCA(n).fit(x[splits == 0])
        xy = pca.transform(x[test_idx])
        if n == 1:
            xy = np.column_stack((xy, np.zeros(len(xy))))
        for row, (j, name) in enumerate(colors):
            sc = axes[row, col].scatter(xy[:, 0], xy[:, 1], c=truth[test_idx, j], s=5, cmap="coolwarm")
            axes[row, col].set_title(f"{method} / {feature}\ncolored by {name}")
            axes[row, col].set_xlabel("train-fitted PC1")
            axes[row, col].set_ylabel("train-fitted PC2")
            fig.colorbar(sc, ax=axes[row, col])
    fig.tight_layout()
    fig.savefig(Path(out)/"latent_test.png", dpi=140)
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for label, _, method, feature in selected:
        subset = scores[(scores.label == label) & (scores.probe == "ridge") & (scores.horizon_frames > 0)]
        for ax, name in zip(axes, ["theta", "velocity"]):
            part = subset[subset.variable == name].sort_values("horizon_frames")
            ax.plot(part.horizon_periods, part.rmse, "o-", label=f"{method}/{feature}")
            ax.fill_between(part.horizon_periods.to_numpy(), part.rmse_ci_low.to_numpy(),
                            part.rmse_ci_high.to_numpy(), alpha=0.12)
            ax.set(xlabel="forecast horizon / reference period", ylabel=f"{name} RMSE",
                   title="Direct forecasts; held-out trajectories")
    for ax, name in zip(axes, ["theta", "velocity"]):
        part = scores[(scores.method == "oracle_state_persistence") & (scores.variable == name)]
        ax.plot(part.horizon_periods, part.rmse, "k--", label="oracle state persistence")
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(Path(out)/"forecast_test.png", dpi=140)
    plt.close(fig)

    # One held-out trajectory, ordered in time: no t-SNE trajectory interpolation.
    tr = int(index[np.flatnonzero(splits == 2)[0], 0])
    mask = index[:, 0] == tr
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for j, name in enumerate(["theta", "velocity"]):
        axes[j].plot(index[mask, 1], truth[mask, j], "k-", linewidth=2, label="truth")
        for label, _, method, feature in selected:
            axes[j].plot(index[mask, 1], current_predictions[label][mask, j], label=f"{method}/{feature}", alpha=0.7)
        axes[j].set(xlabel="frame", ylabel=name, title=f"Held-out trajectory {tr}: linear readout")
        axes[j].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(Path(out)/"state_readout_test.png", dpi=140)
    plt.close(fig)


## Cell 3 — Select dataset and compatible runs


In [ ]:
import pandas as pd
from IPython.display import display, Image
from l1_evaluation import EvalConfig, completed_runs, compare

DATASET_ID = ''
CONDITIONS_TO_EVALUATE = ['L1a']
EVALUATION = EvalConfig(window=5, stride=4, bootstrap_repeats=300, nonlinear_probe=True)
# Empty mapping uses all completed runs matching history/stride; never chooses "best test".
# To compare a prespecified subset, map condition -> list of exact run directory names.
SELECTED_RUN_IDS = {}
DATASET_ID = resolve_dataset_id(DATASET_ID)


## Cell 4 — Fit probes/baselines and evaluate once on held-out groups


In [ ]:
RESULTS = {}
for condition in CONDITIONS_TO_EVALUATE:
    paths = completed_runs(L1_ROOT, DATASET_ID, condition)
    if condition in SELECTED_RUN_IDS:
        requested = set(SELECTED_RUN_IDS[condition])
        found = {p.name for p in paths}
        if not requested <= found:
            raise FileNotFoundError(f'Missing requested runs: {requested-found}')
        paths = [p for p in paths if p.name in requested]
    compatible, methods = [], set()
    for path in paths:
        meta = json.loads((path / 'run.json').read_text())
        training = meta['training']
        if training['window'] == EVALUATION.window and training['eval_stride'] == EVALUATION.stride:
            compatible.append(path); methods.add(meta['method'])
        else:
            print('Skipping incompatible history/stride:', path.name)
    missing = {'nsv', 'infonce', 'bt_temporal', 'bt_same'} - methods
    if missing:
        print('Methods not yet available:', sorted(missing))
    output, scores = compare(L1_ROOT, DATASET_ID, condition, CACHE_DIR, compatible, EVALUATION)
    RESULTS[condition] = (output, scores)
    print(condition, 'outputs:', output)


## Cell 5 — Readouts, prediction curves and held-out trajectories


In [ ]:
for condition, (output, scores) in RESULTS.items():
    print(condition, 'CURRENT STATE READOUTS')
    columns = ['method', 'feature', 'dimension', 'training_seed', 'probe', 'variable',
               'r2', 'rmse', 'rmse_ci_low', 'rmse_ci_high', 'test_groups']
    display(scores.loc[scores.horizon_frames == 0, columns].reset_index(drop=True))
    for name in ['latent_test.png', 'state_readout_test.png', 'forecast_test.png']:
        display(Image(filename=str(output / name)))
    print('All configurations and seeds remain separate in:', output / 'scorecard.csv')


## Interpretation checklist

- L1a: recover velocity direction and decaying motion; a closed ring is not required.
- L1b: compare the same initial conditions and split with L1a, at matched method settings.
- L1c: length is continuous, constant per trajectory; compare visible and hidden rendering.
- Compare encoder h and projector z; InfoNCE normalized z_unit removes radial information.
- ID ~2/3 is a conditional physical reference; noise and sampling can alter estimates.
- Figures display the first available run/low-dimensional feature per method in a fixed
  order for readability. The CSV contains every run; no run is selected by test quality.
- Initial pilot settings are not evidence that any method works. Report actual held-out
  results and seed variation before making scientific claims.
